In [14]:
from pathlib import Path
from statsmodels.tsa.seasonal import seasonal_decompose, STL
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import sys

# путь до корня проекта
PROJECT_ROOT = Path("../../").resolve()

# добавляем в PYTHONPATH
sys.path.append(str(PROJECT_ROOT))

# теперь обычный импорт
#from src.utils.spec_converter import create_feature_spec_template
from src.utils.io import load_feature_names_from_txt

from src.utils.contact_code_utils import exctract_code_pvz, expand_cities_by_comma, get_region

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

In [15]:
# Пути относительно папки notebooks/group_2/
DATA_PATH = Path("../../data/raw/dataset_2025-03-01_2026-03-29_external.csv")
FEATURES_PATH = Path("../../data/feature_groups/features_group_2.txt")
OUTPUT_DIR = Path("../../notebook_outputs/group_2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
XSLX_PATH = Path("../../data/raw/PvzList_rus-2.xlsx")

In [16]:
df = pd.read_csv(DATA_PATH)

C:\Users\Михаил\AppData\Local\Temp\ipykernel_16388\2144582446.py:1: DtypeWarning: Columns (52,76,93,99,100,105,114,115,122,127,128,133,134,140,141,143,144,145,146,147,148,149,150,151,154,156,157,158,159,160,161,162,163,164,167,168,169,170,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


In [17]:
df["contact_Код ПВЗ"] = [exctract_code_pvz(x) for x in df["contact_Код ПВЗ"]]

df["contact_Код ПВЗ"].unique().tolist()

['unknown',
 'MSK2348',
 'KRK3',
 'KMN5',
 'MSK163',
 'MSK599',
 'SPB24',
 'KUR14',
 'ABK14',
 'SPB23',
 'OSTS5',
 'NKHD3',
 'DMT3',
 'ZLG2',
 'BLK9',
 'KZN35',
 'BLK6',
 'SPB1155',
 'SCH37',
 'VLG49',
 'MSK449',
 'USS1',
 'MSK1124',
 'RZN9',
 'PTG3',
 'IZHV53',
 'PRM33',
 'SAM72',
 'RND25',
 'MSK126',
 'MSK340',
 'SAR3',
 'PNZ78',
 'TRC1',
 'MSK2609',
 'MSK1313',
 'MSK2384',
 'KRV15',
 'KST36',
 'MSK2414',
 'VOR1',
 'ULN20',
 'VNG3',
 'YEKB93',
 'DBN9',
 'MSK2362',
 'MSK117',
 'DZZH3',
 'SHCHLK10',
 'EKB8',
 'MKHCH37',
 'LYT11',
 'SPB20',
 'MSK354',
 'MSK303',
 'KLM6',
 'YARS20',
 'PRM24',
 'ART2',
 'MUR5',
 'SPB73',
 'SPB137',
 'MSK2279',
 'IRK78',
 'GVA2',
 'IRK5',
 'KSD5',
 'SVK2',
 'KHME16',
 'KUSHCH2',
 'PRM201',
 'SPB295',
 'MSK51',
 'NBK11',
 'NN38',
 'KLN10',
 'SPB132',
 'VLK26',
 'MSK422',
 'ORN13',
 'PSHK29',
 'IZHV22',
 'BLD16',
 'PRM198',
 'MSK961',
 'MTSHCH87',
 'MSK2370',
 'DMG5',
 'NSK25',
 'KZN13',
 'TUL12',
 'MSK234',
 'MSK568',
 'MSK1071',
 'NVCH2',
 'KRS10',
 'BLSH3

In [18]:
pvz_data = pd.read_excel(XSLX_PATH, sheet_name="Россия", usecols=[0, 1, 2], engine='openpyxl')
pvz_data = expand_cities_by_comma(pvz_data)
pvz_data.head()

,Регион,Город,Код
0,Адыгея,Адыгейск,ADY3
1,Адыгея,Гиагинская,GIG1
2,Адыгея,Каменномостский,KAM49
3,Адыгея,Майкопский район,KAM49
4,Адыгея,Майкоп,MAY3


In [19]:
pvz_dict = {}
for pvz_code, region in zip(pvz_data.iloc[:, 2], pvz_data.iloc[:, 0]):
    match = re.match(r"^([A-Z]+)", str(pvz_code))
    if match:
        pvz_dict[match.group(1)] = region
print(pvz_dict)

{'ADY': 'Адыгея', 'GIG': 'Адыгея', 'KAM': 'Татарстан', 'MAY': 'Ульяновская область', 'TLK': 'Адыгея', 'KHA': 'Саратовская область', 'ENM': 'Адыгея', 'YABL': 'Адыгея', 'ALK': 'Чеченская Республика', 'ALT': 'Татарстан', 'BRN': 'Алтайский край', 'BLKH': 'Алтайский край', 'BSK': 'Алтайский край', 'BLG': 'Республика Башкортостан', 'VOL': 'Саратовская область', 'GAL': 'Саратовская область', 'GRN': 'Пермский край', 'ZAT': 'Алтайский край', 'ZAV': 'Удмуртия', 'ZRS': 'Московская область', 'ZME': 'Алтайский край', 'KAL': 'Ульяновская область', 'KMN': 'Волгоградская область', 'KLY': 'Саратовская область', 'KUL': 'Саратовская область', 'MAM': 'Республика Алтай', 'MIK': 'Ульяновская область', 'NVL': 'Алтайский край', 'NOV': 'Ямало-Ненецкий автономный округ', 'PAV': 'Ульяновская область', 'PAN': 'Ямало-Ненецкий автономный округ', 'POS': 'Саратовская область', 'REB': 'Алтайский край', 'ROD': 'Ульяновская область', 'ROM': 'Саратовская область', 'RBTS': 'Алтайский край', 'RBC': 'Алтайский край', 'SLV':

In [20]:
df["contact_region_pvz"] = df["contact_Код ПВЗ"].apply(
    lambda x: get_region(x, pvz_dict)
)
df["contact_region_pvz"].unique().tolist()

['unknown',
 'Москва',
 'Тульская область',
 'Волгоградская область',
 'Санкт-Петербург',
 'Чеченская Республика',
 'Республика Хакасия',
 'Московская область',
 'Приморский край',
 'Красноярский край',
 'Саратовская область',
 'Татарстан',
 'Краснодарский край',
 'Ростовская область',
 'Рязанская область',
 'Ставропольский край',
 'Удмуртия',
 'Пермский край',
 'Ульяновская область',
 'Пензенская область',
 'Кировская область',
 'Новгородская область',
 'Свердловская область',
 'Нижегородская область',
 'Дагестан',
 'Ярославская область',
 'Ямало-Ненецкий автономный округ',
 'Иркутская область',
 'Томская область',
 'Оренбургская область',
 'Белгородская область',
 'Новосибирская область',
 'Чувашская Республика - Чувашия',
 'Курская область',
 'Омская область',
 'Калининградская область',
 'Челябинская область',
 'Тюменская область',
 'Калужская область',
 'Республика Башкортостан',
 'Республика Карелия',
 'Алтайский край',
 'Вологодская область',
 'Смоленская область',
 'Тверская об

In [21]:
col = "contact_region_pvz"

mask = df[col] != "unknown"
top15 = df.loc[mask, col].value_counts().head(15).index

df[col] = df[col].where(
    df[col].isin(top15) | (df[col] == "unknown"),
    "rare_region"
)

df[col].unique().tolist()

['unknown',
 'Москва',
 'rare_region',
 'Санкт-Петербург',
 'Московская область',
 'Красноярский край',
 'Саратовская область',
 'Татарстан',
 'Краснодарский край',
 'Ростовская область',
 'Ульяновская область',
 'Свердловская область',
 'Нижегородская область',
 'Новосибирская область',
 'Челябинская область',
 'Республика Башкортостан',
 'Ханты-Мансийский автономный округ - Югра']

In [11]:
correct_file = str(PROJECT_ROOT / "bench" / "correct_dataset.csv")
correct_df = pd.read_csv(correct_file)
correct_df["contact_region_pvz"].unique().tolist()

['rare_region',
 'Москва',
 'Санкт-Петербург',
 'Московская область',
 'Красноярский край',
 'Саратовская область',
 'Татарстан',
 'Краснодарский край',
 'Ростовская область',
 'Ульяновская область',
 'Свердловская область',
 'Нижегородская область',
 'Новосибирская область',
 'Челябинская область',
 'Ханты-Мансийский автономный округ - Югра']